In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, roc_curve, confusion_matrix
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline, make_pipeline
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
import os

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
# Step 1: Load the dataset
datapath = os.path.join("/content/drive/MyDrive/TML dataset/")

''' Dataset choose from:
original_fraud_data
essentialCleaning_fraud_data
afterPreprocessed_fraud_data
creditcard
'''
dataset_name = 'original_fraud_data'

df = pd.read_csv(datapath + dataset_name + '.csv')

In [6]:
# Step 2: Define features (X) and label (y) and Drop rows in label where the target is missing
if dataset_name in ['original_fraud_data', 'essentialCleaning_fraud_data', 'afterPreprocessed_fraud_data']:
  print({dataset_name})
  df = df.dropna(subset=['is_fraud'])
  y = df['is_fraud']
  X = df.drop(columns=['is_fraud'])

elif dataset_name == 'creditcard':
  print({dataset_name})
  df = df.dropna(subset=['Class'])
  y = df['Class']
  X = df.drop(columns=['Class'])

else:
  print("Invalid dataset name")



{'original_fraud_data'}


In [7]:
# Step 3: Detect column types
categorical_cols = X.select_dtypes(include='object').columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Step 4: Preprocessor
preprocessor = ColumnTransformer([
    ('cat_encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_cols)
], remainder='passthrough')

# Step 5: Model Pipeline
model_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=200,            # More trees = more stability
        max_depth=None,              # Let trees grow fully (can tune later)
        min_samples_split=2,         # Default; try increasing if overfitting
        max_features='sqrt',         # Good balance for classification
        class_weight='balanced',     # Handle class imbalance
        random_state=42,             # For reproducibility
    ))
])

# Step 6: Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

# Step 7: Train Random Forest  (may need few minutes to execute)
model_pipeline.fit(X_train, y_train)

# Step 8: Predict and evaluate
y_pred = model_pipeline.predict(X_test)
y_proba = model_pipeline.predict_proba(X_test)[:, 1]  # For ROC AUC

# Step 9: Calculate and print metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)

print("Classification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"ROC AUC:   {roc_auc:.4f}")


Classification Report:
               precision    recall  f1-score   support

           0       0.99      1.00      1.00    386751
           1       1.00      0.01      0.01      2252

    accuracy                           0.99    389003
   macro avg       1.00      0.50      0.51    389003
weighted avg       0.99      0.99      0.99    389003

Confusion Matrix:
 [[386751      0]
 [  2236     16]]
Accuracy: 0.9943
Precision: 1.0000
Recall:    0.0071
F1 Score:  0.0141
ROC AUC:   0.9755


In [9]:
# Step 10: Predict and evaluate with threshold (for sensitive FN cases)
y_proba = model_pipeline.predict_proba(X_test)[:, 1]  # For ROC AUC
threshold = 0.1  #can change according to the situation
y_pred_threshold = (y_proba >= threshold).astype(int)

# Print Metrics
print(f"Threshold: {threshold}")
print("Classification Report:\n", classification_report(y_test, y_pred_threshold))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_threshold))
print("Precision: {:.4f}".format(precision_score(y_test, y_pred_threshold)))
print("Recall:    {:.4f}".format(recall_score(y_test, y_pred_threshold)))
print("F1 Score:  {:.4f}".format(f1_score(y_test, y_pred_threshold)))

Threshold: 0.1
Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00    386751
           1       0.78      0.80      0.79      2252

    accuracy                           1.00    389003
   macro avg       0.89      0.90      0.89    389003
weighted avg       1.00      1.00      1.00    389003

Confusion Matrix:
 [[386228    523]
 [   443   1809]]
Precision: 0.7757
Recall:    0.8033
F1 Score:  0.7893
